<a href="https://colab.research.google.com/github/AnilZen/centpy/blob/master/notebooks/Euler_1d.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Euler Equation with CentPy in 1D

### Import packages

In [ ]:
# Install the centpy package
!pip install centpy

In [ ]:
# Import numpy and centpy for the solution
import numpy as np
import centpy

In [ ]:
# Imports functions from matplotlib and setup for the animation
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

### Equation

We solve the Euler equations in 1D

\begin{equation}
\partial_t
\begin{bmatrix} \rho \\ \rho v \\ E \end{bmatrix}
+
\partial_x
\begin{bmatrix} \rho v \\ \rho v^2 +p \\ v (E+p) \end{bmatrix}
= 0
\end{equation}

with the equation of state

\begin{equation}
p = (\gamma-1) \left(E-\frac{1}{2} \rho v^2\right), \qquad \gamma=1.4
\end{equation}

on the domain $(x,t)\in([0,1]\times[0,0.1])$ with initial data for a *Sod shock tube*:

\begin{equation}
(\rho, v, p)_{t=0} =
\begin{cases}
(1,0,1) & \text{if} & 0<x\leq0.5 \\
(0.125, 0, 0.1) & \text{if} & 0.5<x<1
\end{cases}
\end{equation}

and Dirichlet boundary data set by initial data on each boundary. The solution is computed using 400 cells and CFL number 0.75.

In [ ]:
# pars = centpy.Pars1d(x_init=0.0, x_final=1.0, t_final=0.3, dt_out=0.005, J=400, cfl=0.75, scheme="sd2")
# pars.gamma = 1.4

In [ ]:
import numpy as np
import centpy

# Euler equation - Modified Sod
class Euler1d(centpy.Equation1d):
    def initial_data(self):
        u = np.zeros((self.J + 4, 3))

        # Вычисляем точные координаты x для центров ячеек
        x = np.linspace(self.x_init, self.x_final, self.J)
        # Добавляем по 2 фиктивные (ghost) ячейки с каждой стороны
        dx = (self.x_final - self.x_init) / self.J
        x_padded = np.concatenate([[x[0]-2*dx, x[0]-dx], x, [x[-1]+dx, x[-1]+2*dx]])

        # Параметры газа
        rho_0, p_0 = 1.0, 1.0
        alpha = 0.1
        rho_R = 2.66666666667
        p_R = 4.5
        u_R = -1.47901994677

        # Границы зон
        x_0, x_1, x_sw = 0.3, 0.5, 0.8

        # Распределение плотности
        rho = np.where(x_padded < x_0, rho_0,
                np.where(x_padded <= x_1, alpha * rho_0,
                np.where(x_padded < x_sw, rho_0, rho_R)))

        # Распределение скорости
        vel = np.where(x_padded < x_sw, 0.0, u_R)

        # Распределение давления
        p = np.where(x_padded < x_0, p_0,
                np.where(x_padded <= x_1, p_0,
                np.where(x_padded < x_sw, p_0, p_R)))

        # Перевод в консервативные переменные [rho, rho*u, E]
        E = p / (self.gamma - 1.0) + 0.5 * rho * vel**2
        u[:, 0] = rho
        u[:, 1] = rho * vel
        u[:, 2] = E

        return u

    def boundary_conditions(self, u):
        # Граничные условия Неймана (нулевой градиент / zero gradient)
        # Копируем значения из ближайших внутренних ячеек в фиктивные
        u[0, :] = u[2, :]
        u[1, :] = u[2, :]
        u[-1, :] = u[-3, :]
        u[-2, :] = u[-3, :]

    def flux_x(self, u):
        f = np.zeros_like(u)
        # Защита от деления на ноль
        rho = np.maximum(u[:, 0], 1e-10)
        u_x = u[:, 1] / rho
        E = u[:, 2]
        p = (self.gamma - 1.0) * (E - 0.5 * rho * u_x ** 2)

        f[:, 0] = rho * u_x
        f[:, 1] = rho * u_x ** 2 + p
        f[:, 2] = u_x * (E + p)

        return f

    def spectral_radius_x(self, u):
        # Защита от деления на ноль и корня из отрицательного числа
        rho = np.maximum(u[:, 0], 1e-10)
        u_x = u[:, 1] / rho
        E = u[:, 2]
        p = np.maximum((self.gamma - 1.0) * (E - 0.5 * rho * u_x ** 2), 1e-10)

        return np.abs(u_x) + np.sqrt(self.gamma * p / rho)

### Solution

In [ ]:
import time

# Инициализация параметров как в JAXFLUIDS и jax_centpy
pars = centpy.Pars1d(x_init=0.0, x_final=1.0, t_final=0.25, dt_out=0.005, J=16384, cfl=0.45, scheme="sd2")
pars.gamma = 1.4

# Создание уравнения и решателя
eqn = Euler1d(pars)
soln = centpy.Solver1d(eqn)

print("Запуск расчёта классической версии centpy (NumPy)...")

# Замер времени выполнения
t0 = time.time()
soln.solve()
t1 = time.time()

cpu_time = t1 - t0
print(f"Расчёт завершен!")
print(f"Время выполнения (NumPy / CPU): {cpu_time:.4f} секунд")

Запуск расчёта классической версии centpy (NumPy)...
Расчёт завершен!
Время выполнения (NumPy / CPU): 430.9722 секунд


### Animation

In [ ]:
# Animation
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

# First set up the figure, the axis, and the plot element we want to animate
# Создаем 3 графика друг под другом и объединяем ось X
fig, axes = plt.subplots(3, 1, figsize=(8, 10), sharex=True)
fig.tight_layout(pad=4.0)

ax1, ax2, ax3 = axes

# Set the labels (подписи на русском, как на картинке)
ax1.set_ylabel('Плотность', fontsize=16)
ax2.set_ylabel('Скорость', fontsize=16)
ax3.set_ylabel('Давление', fontsize=16)
ax3.set_xlabel('Координата x', fontsize=16)

# Увеличиваем размер цифр на осях
for ax in axes:
    ax.tick_params(axis='both', labelsize=14)
    ax.grid(True) # Включаем сетку

# Axis limits and lines
line_u = []

# Пределы для Плотности
ax1.set_xlim(0, 1.0)
ax1.set_ylim(0.0, 3.0)
line_u.append(ax1.plot([], [], linewidth=1.5, color='#D62728', marker='', linestyle='-')[0])

# Пределы для Скорости
ax2.set_xlim(0, 1.0)
ax2.set_ylim(-2.5, 0.5)
line_u.append(ax2.plot([], [], linewidth=1.5, color='#D62728', marker='', linestyle='-')[0])

# Пределы для Давления
ax3.set_xlim(0, 1.0)
ax3.set_ylim(0.0, 5.0)
line_u.append(ax3.plot([], [], linewidth=1.5, color='#D62728', marker='', linestyle='-')[0])


# animation function.  This is called sequentially
j0 = slice(2,-2)
def animate(i):
    rho = soln.u_n[i,j0,0]
    u_x = soln.u_n[i,j0,1] / soln.u_n[i,j0,0]
    p = (soln.gamma - 1.) * (soln.u_n[i,j0,2] - 0.5 * rho * u_x**2)

    line_u[0].set_data(soln.x[j0], rho)
    line_u[1].set_data(soln.x[j0], u_x)
    line_u[2].set_data(soln.x[j0], p)

    return line_u

plt.close(fig) # Закрываем статичную фигуру
anim = animation.FuncAnimation(fig, animate, frames=soln.Nt, interval=100, blit=True)
HTML(anim.to_html5_video())